In [1]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
from tqdm import tqdm

# notebook is in RAG/notebooks → project root is parent
PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))

print("PROJECT_ROOT:", PROJECT_ROOT)
print("src exists:", (PROJECT_ROOT / "src").exists())


PROJECT_ROOT: C:\Users\Vohita\RAG
src exists: True


In [1]:
from pathlib import Path

DATA_PATH = Path.cwd().parent / "data_legal"
INDEX_PATH = DATA_PATH / "indexes"

INDEX_PATH.mkdir(parents=True, exist_ok=True)



In [3]:
# Load chunks
import pandas as pd
chunks = pd.read_csv(f"{DATA_PATH}/rag_corpus_chunks.csv")

chunks = chunks.reset_index(drop=True)
chunks["row_id"] = chunks.index

chunks.shape

(458, 7)

In [4]:
chunks.head(2)[["row_id", "chunk_id", "chunk_text"]]

,row_id,chunk_id,chunk_text
0,0,LC00001,Company shall not specify the business practic...
1,1,LC00002,In the event that Licensor grants to another V...


In [5]:
# Initialize embedding model
from sentence_transformers import SentenceTransformer
import torch

In [6]:
device = "cuda" if torch.cuda.is_available() else "cpu"
device

'cpu'

In [7]:
embedder = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2",
    device=device
)

In [8]:
# Generate embeddings (BATCHED, SAFE)
BATCH_SIZE = 64

In [10]:
from tqdm import tqdm


In [11]:
all_embeddings = []

texts = chunks["chunk_text"].tolist()

for i in tqdm(range(0, len(texts), BATCH_SIZE)):
    batch = texts[i:i + BATCH_SIZE]

    emb = embedder.encode(
        batch,
        convert_to_numpy=True,
        show_progress_bar=False
    )

    all_embeddings.append(emb)

100%|████████████████████████████████████████████████████████████████████████████████████| 8/8 [00:22<00:00,  2.81s/it]


In [13]:
import numpy as np
embeddings = np.vstack(all_embeddings)
embeddings.shape

(458, 384)

In [14]:
assert embeddings.shape[0] == len(chunks)
assert embeddings.dtype == np.float32 or embeddings.dtype == np.float64

In [15]:
np.save("../data_legal/embeddings.npy", embeddings)


In [16]:
import faiss
faiss.normalize_L2(embeddings)

In [17]:
# Build FAISS index (LEGAL)
import faiss
import numpy as np

# embeddings must be float32
embeddings = embeddings.astype("float32")

# dimension
dim = embeddings.shape[1]

# cosine similarity via inner product
faiss_index = faiss.IndexFlatIP(dim)

# normalize embeddings
faiss.normalize_L2(embeddings)

# add vectors
faiss_index.add(embeddings)

# save LEGAL FAISS index
faiss.write_index(
    faiss_index,
    str(INDEX_PATH / "faiss.index")
)

faiss_index.ntotal


458